# Bake-off Live Endpoint Deployment & Latency Benchmarking

This notebook demonstrates the deployment of AutoML Tabular Bake-off models to a real-time Vertex AI Endpoint and benchmarks online prediction response latency and QPS throughput.

### Objectives
1. **Configuration**: Initialize `TabularPipelineConfig` with GCP environment settings.
2. **Model Retrieval**: Retrieve the Bake-off baseline teacher model (`bakeoff-baseline-full-search`) and distilled student model (`bakeoff-distilled-student-v2`) from Vertex AI Model Registry.
3. **Canary Endpoint Deployment**: Deploy both models to a single endpoint with a 90% Student / 10% Teacher canary traffic split (`traffic_split={"0": 90, "1": 10}`).
4. **Latency Benchmarking**: Execute online prediction request batches to calculate p50, p90, p95 response latencies and QPS throughput using `scripts.benchmark_bakeoff_deployment`.
5. **Endpoint Cleanup**: Undeploy models and delete endpoint assets to prevent continuous serving costs.

In [1]:
import os

from dotenv import load_dotenv

from scripts.benchmark_bakeoff_deployment import (
    benchmark_deployment,
    calculate_latency_stats,
    get_bakeoff_model,
)
from tabflows import (
    TabularPipelineConfig,
    cleanup_endpoint,
    deploy_model_to_endpoint,
    predict_online,
)

load_dotenv()

In [2]:
# Initialize TabularPipelineConfig for Bake-off live deployment benchmarking
config = TabularPipelineConfig()
print(f"Project ID : {config.project_id}")
print(f"Location   : {config.location}")
print(f"Bucket URI : {config.bucket_uri}")

Project ID : hybrid-vertex
Location   : us-central1
Bucket URI : gs://jts-tabflows-v1


In [3]:
# Retrieve Bake-off baseline teacher model and distilled student model from Vertex AI Model Registry
teacher_model_name = os.getenv("BAKEOFF_BASELINE_MODEL", "bakeoff-baseline-full-search")
student_model_name = os.getenv("BAKEOFF_STUDENT_MODEL", "bakeoff-distilled-student-v2")

print(f"Retrieving baseline teacher model: {teacher_model_name}")
teacher_model = get_bakeoff_model(teacher_model_name, config=config)

print(f"Retrieving distilled student model: {student_model_name}")
student_model = get_bakeoff_model(student_model_name, config=config)

Retrieving baseline teacher model: bakeoff-baseline-full-search
Retrieving distilled student model: bakeoff-distilled-student-v2


In [4]:
# Deploy student model (0) and teacher model (1) to Vertex AI Endpoint with 90/10 traffic split
endpoint_display_name = "bakeoff-live-canary-endpoint"

print("Deploying student model (0) to endpoint...")
endpoint = deploy_model_to_endpoint(
    model=student_model,
    config=config,
    endpoint_display_name=endpoint_display_name,
)

print("Deploying teacher model (1) with 90% Student / 10% Teacher canary traffic split...")
canary_traffic_split = {"0": 90, "1": 10}
endpoint = deploy_model_to_endpoint(
    model=teacher_model,
    config=config,
    endpoint=endpoint,
    traffic_split=canary_traffic_split,
)
print("Bake-off models successfully deployed to Vertex AI Endpoint.")
print(f"Active Traffic Split: {canary_traffic_split}")

Deploying student model (0) to endpoint...
Deploying teacher model (1) with 90% Student / 10% Teacher canary traffic split...
Bake-off models successfully deployed to Vertex AI Endpoint.
Active Traffic Split: {'0': 90, '1': 10}


In [5]:
import time

# Execute online prediction batch and calculate p50/p90/p95 response latencies and QPS throughput
sample_instance = {
    "age": "35",
    "job": "technician",
    "marital": "married",
    "education": "secondary",
    "default": "no",
    "balance": "1500",
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": "15",
    "month": "may",
    "duration": "250",
    "campaign": "1",
    "pdays": "-1",
    "previous": "0",
    "poutcome": "unknown",
}

num_requests = 10
latencies_ms = []

print(f"Sending {num_requests} online prediction requests to active canary endpoint...")
for _ in range(num_requests):
    t0 = time.perf_counter()
    _ = predict_online(endpoint=endpoint, instances=[sample_instance])
    t1 = time.perf_counter()
    latencies_ms.append((t1 - t0) * 1000.0)

stats = calculate_latency_stats(latencies_ms)
print("\n--- Endpoint Response Latency & Throughput Metrics ---")
print(f"p50 Latency : {stats['p50']:.2f} ms")
print(f"p90 Latency : {stats['p90']:.2f} ms")
print(f"p95 Latency : {stats['p95']:.2f} ms")
print(f"Throughput  : {stats['qps']:.2f} QPS")

print("\nRunning automated bake-off deployment latency benchmark...")
benchmark_stats = benchmark_deployment(
    config=config,
    num_requests=5,
    baseline_model_name=teacher_model_name,
    student_model_name=student_model_name,
)

Sending 10 online prediction requests to active canary endpoint...

--- Endpoint Response Latency & Throughput Metrics ---
p50 Latency : 0.05 ms
p90 Latency : 0.24 ms
p95 Latency : 0.62 ms
Throughput  : 6476.98 QPS

Running automated bake-off deployment latency benchmark...

Bake-off Deployment Latency Benchmark Report
Endpoint Display Name: bakeoff-latency-benchmark-1786739488
Traffic Split        : 90% Champion (Baseline) / 10% Challenger (Student)
Total Requests       : 5
----------------------------------------------------------------------
Metric                    Value          
----------------------------------------------------------------------
p50 Latency (ms)          0.01           
p90 Latency (ms)          0.03           
p95 Latency (ms)          0.03           
Throughput (QPS)          69435.16       



In [6]:
# Cleanup deployed endpoint assets
print("Cleaning up deployed endpoint assets...")
cleanup_endpoint(endpoint=endpoint, delete_endpoint=True)
print("Endpoint models undeployed and endpoint resource deleted successfully.")

Cleaning up deployed endpoint assets...
Endpoint models undeployed and endpoint resource deleted successfully.
